# Referências — AI Governance in Brazilian Universities

**Projeto:** `ai-governance-brazilian-universities`  
**Corpus:** 20 PDFs em `data/processed/documentos/`  

Este notebook tem duas partes:

1. **Exportação dos textos** — extrai o conteúdo de cada PDF e salva como `.txt` em `data/processed/textos/`
2. **Extração de referências** — aplica padrões regex sobre os textos e exporta um CSV no estilo KWIC (Key Word in Context), para revisão manual

O CSV exportado (`outputs/referencias_kwic.csv`) será revisado manualmente antes de qualquer visualização em rede.

---

In [ ]:
# ── Dependências ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import pandas as pd
import pdfplumber

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path("..")
CSV_PATH   = ROOT / "data" / "raw" / "levantamento_ia_universidades_brasileiras.csv"
PDF_DIR    = ROOT / "data" / "processed" / "documentos"
TXT_DIR    = ROOT / "data" / "processed" / "textos"
OUTPUT_DIR = ROOT / "outputs"

TXT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ambiente pronto.")

---
## Parte 1 — Exportação dos textos extraídos

In [ ]:
def extrair_texto_pdf(caminho: Path, max_paginas: int = 30) -> str:
    """Extrai texto de um PDF com pdfplumber."""
    try:
        with pdfplumber.open(caminho) as pdf:
            paginas = pdf.pages[:max_paginas]
            texto = " ".join((p.extract_text() or "") for p in paginas)
        return texto.strip()
    except Exception as e:
        print(f"  ERRO em {caminho.name}: {e}")
        return ""


# Carrega o CSV para obter a lista de arquivos
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df_pdfs = df[df["arquivo"].notna() & (df["arquivo"].str.strip() != "")].reset_index(drop=True)

print(f"PDFs no levantamento: {len(df_pdfs)}")
print(f"Destino dos .txt: {TXT_DIR}\n")
print("Extraindo e salvando...")

registros = []

for _, row in df_pdfs.iterrows():
    pdf_path = PDF_DIR / row["arquivo"]
    txt_nome = Path(row["arquivo"]).stem + ".txt"
    txt_path = TXT_DIR / txt_nome

    if pdf_path.exists():
        texto = extrair_texto_pdf(pdf_path)
        txt_path.write_text(texto, encoding="utf-8")
        status = f"{len(texto):>7,} chars → {txt_nome}"
    else:
        texto = ""
        status = "ARQUIVO NÃO ENCONTRADO"

    registros.append({
        "arquivo_pdf": row["arquivo"],
        "arquivo_txt": txt_nome if pdf_path.exists() else "",
        "instituicao": row["instituicao"],
        "ano":         row["ano"],
        "n_chars":     len(texto),
        "texto":       texto,
    })
    print(f"  {row['arquivo']:<55} {status}")

corpus = pd.DataFrame(registros)
corpus_valido = corpus[corpus["n_chars"] > 100].reset_index(drop=True)

print(f"\nTextos exportados com sucesso: {len(corpus_valido)} de {len(df_pdfs)}")
print(f"Arquivos .txt salvos em: {TXT_DIR}")

---
## Parte 2 — Extração de referências (KWIC)

A extração usa expressões regulares organizadas em categorias:

| Categoria | Exemplos |
|---|---|
| Legislação brasileira | Lei nº 13.709, Decreto nº 9.854, Resolução nº 7 |
| Documentos internacionais | UNESCO, OCDE, EU AI Act, Parlamento Europeu |
| Siglas institucionais | LGPD, PL 2338, Marco Civil |
| Referências bibliográficas | padrões ABNT — autor, ano entre parênteses |

Cada ocorrência gera uma linha com **contexto esquerdo**, **termo encontrado** e **contexto direito** — estilo KWIC do AntConc.

In [ ]:
# ── Padrões de referência ─────────────────────────────────────────────────────

PADROES = {
    # Legislação brasileira
    "Lei": (
        r"Lei\s+(?:Complementar\s+)?n[oº°]?\s*[\d\.]+(?:\/\d+)?",
        "legislacao_br"
    ),
    "Decreto": (
        r"Decreto(?:-Lei)?\s+n[oº°]?\s*[\d\.]+(?:\/\d+)?",
        "legislacao_br"
    ),
    "Resolucao": (
        r"Resolu[çc][aã]o\s+(?:CNE\/CES\s+)?n[oº°]?\s*[\d\.]+(?:\/\d+)?",
        "legislacao_br"
    ),
    "Portaria": (
        r"Portaria\s+(?:Normativa\s+)?n[oº°]?\s*[\d\.]+(?:\/\d+)?",
        "legislacao_br"
    ),
    "Instrucao_Normativa": (
        r"Instru[çc][aã]o\s+Normativa\s+n[oº°]?\s*[\d\.]+(?:\/\d+)?",
        "legislacao_br"
    ),
    "Emenda_Constitucional": (
        r"Emenda\s+Constitucional\s+n[oº°]?\s*[\d\.]+(?:\/\d+)?",
        "legislacao_br"
    ),

    # Siglas e marcos legais brasileiros
    "LGPD": (
        r"\bLGPD\b",
        "sigla_br"
    ),
    "PL_2338": (
        r"PL\s*2[\.\\ ]?338(?:\/\d+)?",
        "sigla_br"
    ),
    "Marco_Civil": (
        r"Marco\s+Civil\s+da\s+Internet",
        "sigla_br"
    ),
    "Constituicao_Federal": (
        r"Constitui[çc][aã]o\s+Federal(?:\s+de\s+1988)?",
        "sigla_br"
    ),
    "LDB": (
        r"\bLDB\b|Lei\s+de\s+Diretrizes\s+e\s+Bases",
        "sigla_br"
    ),

    # Documentos e organismos internacionais
    "UNESCO": (
        r"UNESCO(?:\s+\(\d{4}\))?",
        "internacional"
    ),
    "OCDE": (
        r"\bOCDE\b|\bOECD\b",
        "internacional"
    ),
    "EU_AI_Act": (
        r"EU\s+AI\s+Act|AI\s+Act|Regulamento\s+(?:Europeu\s+)?de\s+Intelig[eê]ncia\s+Artificial",
        "internacional"
    ),
    "Parlamento_Europeu": (
        r"Parlamento\s+Europeu|Uni[aã]o\s+Europeia",
        "internacional"
    ),
    "ONU": (
        r"\bONU\b|Na[çc][oõ]es\s+Unidas",
        "internacional"
    ),
    "Conselho_Europa": (
        r"Conselho\s+da\s+Europa|Council\s+of\s+Europe",
        "internacional"
    ),

    # Referências bibliográficas (estilo ABNT — autor + ano)
    "Ref_ABNT": (
        r"[A-ZÁÉÍÓÚÀÂÊÔÃÕÇ][A-ZÁÉÍÓÚÀÂÊÔÃÕÇ\s]{2,40},\s+\d{4}",
        "bibliografia"
    ),
}

print(f"Padrões carregados: {len(PADROES)}")

In [ ]:
# ── Extração KWIC ─────────────────────────────────────────────────────────────

CONTEXTO_CHARS = 120  # caracteres à esquerda e à direita do termo


def extrair_kwic(texto: str, padrao: str, contexto: int = CONTEXTO_CHARS):
    """Retorna lista de (contexto_esq, termo, contexto_dir) para cada ocorrência."""
    resultados = []
    for m in re.finditer(padrao, texto, flags=re.IGNORECASE):
        inicio = m.start()
        fim    = m.end()
        esq    = texto[max(0, inicio - contexto):inicio].replace("\n", " ").strip()
        termo  = m.group()
        dir_   = texto[fim:fim + contexto].replace("\n", " ").strip()
        resultados.append((esq, termo, dir_))
    return resultados


linhas = []

for _, row in corpus_valido.iterrows():
    texto = row["texto"]
    for nome_padrao, (regex, categoria) in PADROES.items():
        ocorrencias = extrair_kwic(texto, regex)
        for esq, termo, dir_ in ocorrencias:
            linhas.append({
                "instituicao":      row["instituicao"],
                "ano":              row["ano"],
                "arquivo_pdf":      row["arquivo_pdf"],
                "categoria_ref":    categoria,
                "padrao":           nome_padrao,
                "termo_encontrado": termo,
                "contexto_esq":     esq,
                "contexto_dir":     dir_,
                # Campos para preenchimento manual
                "referencia_limpa": "",  # versão normalizada do termo
                "valido":           "",  # S / N — é de fato uma referência?
                "nota":             "",  # observação livre
            })

kwic = pd.DataFrame(linhas)

print(f"Total de ocorrências extraídas: {len(kwic)}")
print()
print(kwic.groupby(["categoria_ref", "padrao"])["termo_encontrado"].count().to_string())

### Amostra do resultado

In [ ]:
# Exibe 3 exemplos por categoria para inspeção rápida
for cat in kwic["categoria_ref"].unique():
    print(f"\n{'─'*80}")
    print(f"CATEGORIA: {cat.upper()}")
    print(f"{'─'*80}")
    amostra = kwic[kwic["categoria_ref"] == cat].head(3)
    for _, r in amostra.iterrows():
        print(f"  [{r['instituicao']} {r['ano']}]")
        print(f"  ...{r['contexto_esq']} >>> {r['termo_encontrado']} <<< {r['contexto_dir']}...")
        print()

### Exportação do CSV para revisão manual

In [ ]:
saida = ROOT / "data" / "processed" / "referencias_kwic.csv"
kwic.to_csv(saida, index=False, encoding="utf-8-sig")
print(f"CSV exportado: {saida}")
print(f"Linhas: {len(kwic)}")
print()
print("Colunas para preenchimento manual:")
print("  referencia_limpa — versão normalizada do termo (ex: 'Lei nº 13.709/2018 (LGPD)')")
print("  valido           — S se é de fato uma referência, N se é falso positivo")
print("  nota             — observação livre")